# iNaturalist other-plant observations — Coimbra district

**Purpose.** Download, look at and clean the plant observations that
are not *Acacia*. They become the other-plant classes: they were made
by the same people, in the same places, as the *Acacia* presences, so
the model learns *Acacia* vs. other vegetation rather than "visited
vs. not visited" (Phillips et al., 2009).

**Data.** Every plant observation in the iNaturalist place "Coimbra,
PT" (the district), excluding *Acacia*, all years
(`data/raw/citizen_science/`, see `data/README.md`).

**Download:** 1. observations; 2. obscured coordinates.

**Part 1 — exploration:** 3. when; 4. presences vs. other plants
(map); 5. footprint overlap on a 1 km grid.

**Part 2 — cleaning:** 6. keep dated, non-obscured observations;
7. anonymise the main observer; 8. keep only research grade;
9. positional accuracy; 10. remove other plants in pixels with
*A. dealbata*; 11. group species into classes; 12. herbaceous plants
only where herbs are the top layer, no shrubs under forest; 13. one observation per pixel; 14. burned-area
flag; 15. save.

Needs `acacia_observations.ipynb` to have run first (it reads the
cleaned *Acacia* file to compare).

## 0. Setup

- `PLACE_ID = 8422`: the iNaturalist place for the Coimbra district;
- `ACACIA = 47452` and `PLANTAE = 47126`: iNaturalist taxon ids for
  the genus *Acacia* and for all plants;
- `REF_YEAR = 2023`: the year all observations are treated as;
- EPSG:3763 (PT-TM06, metres), so distances and 1 km cells are
  correct.

In [ ]:
import re
import subprocess
import sys
import time
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from shapely.geometry import shape

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from src.burned_areas import burned_between, load_burned_areas
from src.grid import pixel_id

RAW_CS = ROOT / "data" / "raw" / "citizen_science"
INTERIM_CS = ROOT / "data" / "interim" / "citizen_science"
PLACE_ID = 8422          # iNaturalist "Coimbra, PT" (district)
ACACIA = 47452           # genus Acacia
PLANTAE = 47126          # kingdom Plantae
REF_YEAR = 2023          # all observations are treated as this year

place = requests.get(f"https://api.inaturalist.org/v1/places/{PLACE_ID}", timeout=60).json()["results"][0]
district = gpd.GeoSeries([shape(place["geometry_geojson"])], crs=4326).to_crs(3763).iloc[0]

def base(ax, title):
    gpd.GeoSeries([district], crs=3763).boundary.plot(ax=ax, color="black", linewidth=0.8)
    ax.set_title(title); ax.set_axis_off()

## 1. Download: observations

**What it does.** Asks the iNaturalist API for every plant observation
in the Coimbra district (`place_id=8422`), excluding *Acacia*, and
saves them as CSV files in `data/raw/citizen_science/`.

**How.**
- The download is split into three date ranges (before 2021,
  2021–2025, from 2026), because 2021–2025 was fetched first and the
  other years later. Each range is one file.
- The API returns 200 observations per page. Pages are requested in
  order of id, waiting 1 s between requests (iNaturalist's rate
  limit).
- It is **resumable**: each page is appended to a `.partial.csv` file,
  and if the download stops it restarts from the last saved id. When
  a range finishes, the partial file is renamed.
- If a file already exists, it is not downloaded again.

**Output.** A table with the number of observations per year.

In [ ]:
# Downloaded in date ranges; 2021–2025 was fetched first, the other ranges later.
BG_RANGES = {"before_2021": (None, "2020-12-31"),
             "2021_2025": ("2021-01-01", "2025-12-31"),
             "from_2026": ("2026-01-01", None)}
bg_file = lambda key: RAW_CS / f"inaturalist_plantae_no_acacia_coimbra_{key}.csv"

def date_params(key):
    d1, d2 = BG_RANGES[key]
    return {k: v for k, v in dict(d1=d1, d2=d2).items() if v}

def download_background(key):
    """Resumable: appends each page to a partial file and restarts from the last id saved."""
    partial = bg_file(key).with_suffix(".partial.csv")
    url = "https://api.inaturalist.org/v1/observations"
    last_id = int(pd.read_csv(partial, usecols=["id"])["id"].max()) if partial.exists() else 0
    params = dict(place_id=PLACE_ID, taxon_id=PLANTAE, without_taxon_id=ACACIA, **date_params(key),
                  order_by="id", order="asc", per_page=200, id_above=last_id)
    while True:
        r = requests.get(url, params=params, timeout=60)
        r.raise_for_status()
        res = r.json()["results"]
        if not res:
            break
        rows = []
        for o in res:
            if not o.get("geojson"):
                continue
            lon, lat = o["geojson"]["coordinates"]
            t = o.get("taxon") or {}
            rows.append(dict(id=o["id"], observed_on=o.get("observed_on"),
                             taxon_id=t.get("id"), taxon_name=t.get("name"), taxon_rank=t.get("rank"),
                             latitude=lat, longitude=lon,
                             positional_accuracy=o.get("positional_accuracy"),
                             quality_grade=o.get("quality_grade"),
                             user_login=(o.get("user") or {}).get("login"),
                             license_code=o.get("license_code")))
        pd.DataFrame(rows).to_csv(partial, mode="a", header=not partial.exists(), index=False)
        params["id_above"] = res[-1]["id"]
        time.sleep(1)  # iNaturalist asks for <= 60 requests/min
    partial.rename(bg_file(key))

for key in BG_RANGES:
    if not bg_file(key).exists():
        download_background(key)
bg_raw = pd.concat([pd.read_csv(bg_file(key)) for key in BG_RANGES], ignore_index=True)
bg_raw.groupby(pd.to_datetime(bg_raw["observed_on"], errors="coerce").dt.year).size()

**Result.** 120 479 observations. Like *Acacia*, they grow fast
after 2019: most are from 2021–2026.

## 2. Download: obscured coordinates

**What it does.** Downloads the ids of observations whose location is
hidden, and stores them in `*_obscured_ids.csv` (one file per range).

**Why.** When an observer or a species has geoprivacy set to
"obscured" or "private", iNaturalist moves the point to a random spot
inside a ~0.2° cell (several km). Those coordinates are useless for
10 m pixels. They are excluded in step 6.

**How.** Two queries per range, one for user geoprivacy and one for
taxon geoprivacy, returning ids only.

**Output.** The number of obscured ids.

In [ ]:
obscured_file = lambda key: RAW_CS / f"inaturalist_plantae_no_acacia_coimbra_{key}_obscured_ids.csv"

def download_obscured_ids(key):
    url = "https://api.inaturalist.org/v1/observations"
    ids = set()
    for privacy in [dict(geoprivacy="obscured,private"), dict(taxon_geoprivacy="obscured,private")]:
        params = dict(place_id=PLACE_ID, taxon_id=PLANTAE, without_taxon_id=ACACIA, **date_params(key),
                      order_by="id", order="asc", per_page=200, id_above=0, only_id="true", **privacy)
        while True:
            r = requests.get(url, params=params, timeout=60)
            r.raise_for_status()
            res = r.json()["results"]
            if not res:
                break
            ids.update(o["id"] for o in res)
            params["id_above"] = res[-1]["id"]
            time.sleep(1)
    pd.DataFrame({"id": sorted(ids)}).to_csv(obscured_file(key), index=False)

for key in BG_RANGES:
    if not obscured_file(key).exists():
        download_obscured_ids(key)
obscured_ids = set().union(*(set(pd.read_csv(obscured_file(key))["id"]) for key in BG_RANGES))
len(obscured_ids)

**Result.** 9 903 observations (8%) have obscured coordinates.

# Part 1 — Exploration

## 3. When

Observations per year (from the download above). The comparisons below
leave out obscured coordinates, which are displaced by several km.

In [ ]:
bg_raw["date"] = pd.to_datetime(bg_raw["observed_on"], errors="coerce")
bg_raw["year"] = bg_raw["date"].dt.year
print(f"{len(bg_raw):,} observations, {bg_raw.year.min():.0f}–{bg_raw.year.max():.0f}; "
      f"{bg_raw.year.isna().sum()} without a date; {bg_raw.id.isin(obscured_ids).sum():,} obscured")
bg_raw.year.value_counts().sort_index().loc[2010:].plot.bar(color="tab:gray", figsize=(10, 3), title="Other plants per year (from 2010)")
plt.tight_layout()

## 4. Presences vs. other plants

**What:** *A. dealbata* presences (red, cleaned file) on top of other-plant
observations (grey), in three panels: all observers, Observer A only,
and everyone else. Observer A is the person with the most *Acacia*
observations (anonymised).
**Why:** using other plants only works if they come from the same
places as the presences. The map shows whether that holds, and how
much of it depends on Observer A.

In [ ]:
acacia = pd.read_csv(INTERIM_CS / "inat_dealbata_clean.csv")
acacia = gpd.GeoDataFrame(acacia, geometry=gpd.points_from_xy(acacia.longitude, acacia.latitude), crs=4326).to_crs(3763)
bg = bg_raw[bg_raw["year"].notna() & ~bg_raw["id"].isin(obscured_ids)].copy()
bg = gpd.GeoDataFrame(bg, geometry=gpd.points_from_xy(bg.longitude, bg.latitude), crs=4326).to_crs(3763)

# Observer A = the person with the most Acacia observations (username not shown)
acacia_raw = pd.read_csv(RAW_CS / "inaturalist_acacia_coimbra_pilot.csv")
dominant = acacia_raw.loc[~acacia_raw["coordinates_obscured"], "user_login"].value_counts().index[0]
bg["observer"] = np.where(bg["user_login"] == dominant, "Observer A", "Others")

groups = [("All observers", acacia, bg),
          ("Observer A", acacia[acacia.observer == "Observer A"], bg[bg.observer == "Observer A"]),
          ("Others", acacia[acacia.observer == "Others"], bg[bg.observer == "Others"])]
fig, axes = plt.subplots(1, 3, figsize=(18, 7), sharex=True, sharey=True)
for ax, (label, p, b) in zip(axes, groups):
    b.plot(ax=ax, markersize=0.5, color="tab:gray", alpha=0.3, label=f"other plants (n={len(b)})")
    p.plot(ax=ax, markersize=2, color="tab:red", alpha=0.7, label=f"Acacia (n={len(p)})")
    base(ax, label); ax.legend(loc="lower right", markerscale=5)
plt.tight_layout()

**Result.** The other plants (grey) cover the whole district, and the
*Acacia* presences (red) sit inside areas that also have other
plants, for Observer A and for everyone else.

## 5. Footprint overlap on a 1 km grid

**What it does.** Divides the district into 1 km × 1 km cells and, for
each group (all, Observer A, others), counts:
- the cells with *Acacia* and the cells with other plants;
- `pct_acacia_cells_with_background`: of the cells with *Acacia*, the
  share that also have other plants. High = the other plants cover
  the places where *Acacia* was seen;
- `pct_background_cells_with_acacia`: of the cells with other plants,
  the share that also have *Acacia*. Low = other plants cover much
  more territory than *Acacia*;
- `ratio_bg_per_presence`: other-plant observations per *Acacia*
  presence.

In [ ]:
def cells(df, size=1000):
    return set(zip((df.geometry.x // size).astype(int), (df.geometry.y // size).astype(int)))

rows = []
for label, p, b in groups:
    cp, cb = cells(p), cells(b)
    rows.append(dict(group=label, acacia=len(p), background=len(b),
                     ratio_bg_per_presence=round(len(b) / max(len(p), 1), 1),
                     acacia_cells_1km=len(cp), background_cells_1km=len(cb),
                     pct_acacia_cells_with_background=round(100 * len(cp & cb) / max(len(cp), 1), 1),
                     pct_background_cells_with_acacia=round(100 * len(cp & cb) / max(len(cb), 1), 1)))
pd.DataFrame(rows)

**Result.**
- 32 other-plant observations per *A. dealbata* presence.
- 96.6% of the 1 km cells with *A. dealbata* also have other plants
  (97.5% for Observer A, 93.2% for the others): the other plants
  cover the places where *A. dealbata* was seen.
- Only 16.3% of the cells with other plants have *A. dealbata*: the
  other plants cover much more territory.
- Observer A also records other plants: 36 080 observations (33%).

# Part 2 — Cleaning (preprocessing)

The same steps as for *Acacia* (see `acacia_observations.ipynb`),
saved to `data/interim/citizen_science/inat_other_plants_clean.csv`.

## 6. Keep dated, non-obscured observations

Already applied above when building `bg` (section 4): observations
without a date and with obscured coordinates are left out.

In [ ]:
print(f"{len(bg_raw):,} → {len(bg):,} (dropped {bg_raw.year.isna().sum()} without a date, "
      f"{bg_raw.id.isin(obscured_ids).sum():,} obscured)")

## 7. Anonymise the main observer

Done in section 4: `observer` is "Observer A" (the same person as in
the *Acacia* file) or "Others"; the username is not saved.

In [ ]:
bg["observer"].value_counts()

## 8. Keep only research grade

**What:** keep observations with `quality_grade = research`; drop
`casual` and `needs_id`.
**Why (decided 2026-10-29, see `thesis/notes.md`):** the same rule as
for the *A. dealbata* presences (`acacia_observations.ipynb`, step
11). Other plants only work as a comparison if they were recorded by
the same process as the presences (target-group background, Phillips
et al., 2009). With research grade on one side only, the model could
learn how observations were made (photo-less trips, cultivated garden
plants) instead of the plant.

**Future work:** the sensitivity check with and without `casual` and
`needs_id` is run on both sides at once. Many `needs_id` are
identified to genus, which may be enough to group them into classes:
they can be recovered from the raw files if a class lacks data.

In [ ]:
print(pd.crosstab(bg["quality_grade"], bg["observer"], margins=True))
before = len(bg)
bg = bg[bg["quality_grade"] == "research"]
print(f"\n{before:,} → {len(bg):,} (dropped {before - len(bg):,} casual or needs_id)")
n_pres = len(acacia)
print(f"other plants per A. dealbata presence: {len(bg) / n_pres:.1f}; Observer A: {(bg.observer == 'Observer A').mean():.1%}")

# Does the footprint still match after the filter? (same measures as section 5)
for label, p, b in [("All observers", acacia, bg),
                    ("Observer A", acacia[acacia.observer == "Observer A"], bg[bg.observer == "Observer A"]),
                    ("Others", acacia[acacia.observer == "Others"], bg[bg.observer == "Others"])]:
    cp, cb = cells(p), cells(b)
    print(f"{label}: {100 * len(cp & cb) / len(cp):.1f}% of A. dealbata cells have other plants; "
          f"{100 * len(cp & cb) / len(cb):.1f}% of other-plant cells have A. dealbata")

**Result.** 60 630 research-grade observations remain; 49 946 are
dropped (12 458 `casual`, 37 488 `needs_id`). There are still 20.2
other-plant observations per (final) *A. dealbata* presence, and the
footprint still matches: 94.3% of the 1 km cells with *A. dealbata*
have other plants (96.7% for Observer A, 88.3% for the others).
Observer A made 29.0% of them.

## 9. Positional accuracy

**What:** the same rule as for the presences
(`acacia_observations.ipynb`, step 12, where the test behind it is):
drop observations with a known `positional_accuracy` above 30 m; keep
those without a value, marked with `accuracy_missing`.
**Why the same rule:** as for the quality grade, both sides must go
through the same filters.

In [ ]:
print(f"without a value: {bg.positional_accuracy.isna().mean():.1%} "
      f"(Observer A: {bg.loc[bg.observer == 'Observer A', 'positional_accuracy'].isna().mean():.1%}, "
      f"others: {bg.loc[bg.observer == 'Others', 'positional_accuracy'].isna().mean():.1%})")
before = len(bg)
bg["accuracy_missing"] = bg["positional_accuracy"].isna()
bg = bg[bg["accuracy_missing"] | (bg["positional_accuracy"] <= 30)]
print(f"{before:,} → {len(bg):,} (dropped {before - len(bg):,} with accuracy > 30 m); "
      f"without a value (kept): {bg.accuracy_missing.sum():,}")
print(f"other plants per A. dealbata presence: {len(bg) / len(acacia):.1f}; Observer A: {(bg.observer == 'Observer A').mean():.1%}")

**Result.** As for the presences, missing values are mostly
Observer A's (95.9% of theirs, 19.3% of the others'). 12 505
observations dropped (accuracy > 30 m); 48 125 remain, 25 163 of them
without a value (kept). 16.0 other plants per presence; Observer A's
share rises to 36.2%.

## 10. Remove other plants in pixels with *A. dealbata*

**What:** each observation gets a `pixel` id (same 10 m grid as the
presences, `src/grid.py`). Other plants in a pixel that also holds a
cleaned *A. dealbata* presence are dropped; the presence stays.
**Why (decided 2026-10-29, see `thesis/notes.md`):** otherwise the
same pixel says "*A. dealbata*" and "not *A. dealbata*", which
confuses training. In the field this is normal (other plants under or
next to the acacias). The presence stays because it is the plant to
map and it was seen there. This is the smallest version of the
supervisor's segmentation mask (one pixel); a larger zone around each
acacia waits for his answer.

Repeats within the other plants are handled in step 13, after the
species are grouped into classes.

In [ ]:
bg["pixel"] = pixel_id(bg)
conflict = bg["pixel"].isin(set(acacia["pixel"]))
print(f"pixels with both: {bg.loc[conflict, 'pixel'].nunique()}; other plants in them: {conflict.sum()}")
before = len(bg)
bg = bg[~conflict]
print(f"{before:,} → {len(bg):,}")
shared = bg["pixel"].duplicated(keep=False)
print(f"other plants sharing a pixel (left for after the classes): {shared.mean():.1%}; "
      f"other plants per A. dealbata presence: {len(bg) / len(acacia):.1f}")

**Result.** 443 pixels held both; their 772 other-plant observations
are dropped (48 125 → 47 353). 40.6% of the remaining other plants
still share a pixel with another other plant, handled in step 13. 15.8 other plants per presence.

## 11. Group species into classes

**What:** each observation gets a `group`, the class the model will
learn. There are 1 549 taxa, too many to be classes, so they are
grouped into 8 classes. The groups come from three sources, not from
hand-made lists:

- **Growth form** (tree, shrub, herb), **aquatic** and **maximum
  height**: GIFT, the Global Inventory of Floras and Traits, version
  3.2 (Weigelt et al., 2020; Denelle et al., 2023), downloaded from
  its API. When a species has no growth form in GIFT, the most common
  growth form of its genus in GIFT is used.
- **Invasive:** the national list of invasive species for mainland
  Portugal, Annex II of Decreto-Lei n.º 92/2019 (ICNF PDF). Species
  marked "\*" are invasive only in Madeira and are left out. Names
  are matched on the binomial, with a short table of synonyms (the
  list uses older names). *Hakea decurrens* subsp. *physocarpa* counts
  as the listed *H. sericea*: the EPPO risk analysis (2024) found that
  the invasive *Hakea* populations sampled in Portugal are this taxon,
  long recorded as *H. sericea*.
- **Conifers** and **non-vascular plants:** the iNaturalist taxonomy
  (class Pinopsida; phylum Tracheophyta).

Rules, in order (the first that applies wins):

| Group | Rule |
|---|---|
| aquatic | GIFT aquatic |
| eucalyptus | genus *Eucalyptus* |
| pines and other conifers | Pinopsida and GIFT tree |
| oaks | genus *Quercus* and GIFT tree |
| other large invasives | on the national list, and GIFT tree or shrub, or a non-climbing plant taller than 2 m (GIFT maximum height) |
| other trees | GIFT tree |
| shrubs | GIFT shrub |
| herbaceous | the rest (GIFT herb, "other", or no growth form) |

Mosses, liverworts and algae (non-vascular) are dropped: they are
tiny or marine and say nothing about what a 10 m pixel looks like.

**Why (decided 2026-10-30, to confirm with the supervisor, see
`thesis/notes.md`):** growth forms follow the FAO Land Cover
Classification System, which separates trees, shrubs and herbaceous
vegetation (Di Gregorio & Jansen, 2000). Eucalyptus and pines are the
main forests of the district; oaks the main native trees; the large
invasives are the plants most like *Acacia*. Grouping species into
classes follows Ball et al. (2026), who mapped tree species *and
groups* as classes.

**Limitations:** GIFT has errors (it lists *Hakea decurrens* as a
climber; so the climber rule is not applied to trees and shrubs) and
classes some species differently from their local form (*Quercus
coccifera* as a tree). The 2 m threshold for large invasive herbs
(*Cortaderia*, *Arundo*, *Phytolacca*) is my choice, not from a
source. The synonym table was written by hand.

In [ ]:
GIFT = ROOT / "data" / "external" / "auxiliary" / "gift"
GIFT_API = "https://gift.uni-goettingen.de/api/extended/index3.2.php"   # GIFT version 3.2
GIFT_TRAITS = {"1.2.1": "growth_form", "1.4.1": "climber", "1.6.2": "max_height", "1.7.1": "aquatic"}

def gift_pages(**params):
    rows = []
    while True:
        r = requests.get(GIFT_API, params={**params, "startat": len(rows)}, timeout=300)
        r.raise_for_status()
        rows += r.json()
        if len(r.json()) < 10_000:
            return pd.DataFrame(rows)
        time.sleep(1)

GIFT.mkdir(parents=True, exist_ok=True)
if not (GIFT / "gift_species.csv").exists():
    gift_pages(query="species")[["work_ID", "work_species"]].to_csv(GIFT / "gift_species.csv", index=False)
for tid in GIFT_TRAITS:
    if not (GIFT / f"gift_trait_{tid}.csv").exists():
        gift_pages(query="traits", traitid=tid, biasref=1, biasderiv=1)[["work_ID", "trait_value", "agreement"]] \
            .to_csv(GIFT / f"gift_trait_{tid}.csv", index=False)
gift = pd.read_csv(GIFT / "gift_species.csv")
for tid, col in GIFT_TRAITS.items():
    gift[col] = gift["work_ID"].map(pd.read_csv(GIFT / f"gift_trait_{tid}.csv").set_index("work_ID")["trait_value"])
gift["genus"] = gift["work_species"].str.split().str[0]

INVASIVE_PDF = ROOT / "data" / "external" / "auxiliary" / "icnf_invasive_list" / "lista_nacional_especies_invasoras_anexo_II_DL92_2019.pdf"
if not INVASIVE_PDF.exists():
    INVASIVE_PDF.parent.mkdir(parents=True, exist_ok=True)
    INVASIVE_PDF.write_bytes(requests.get("https://www.icnf.pt/api/file/doc/cf455e5a84ab1d96", timeout=60).content)
text = subprocess.run(["pdftotext", "-layout", str(INVASIVE_PDF), "-"], capture_output=True, text=True).stdout
plants = text.split("\nPlantas\n")[1].split("\nMoluscos\n")[0]
invasive_list = {" ".join(l.split()[:2]) for l in map(str.strip, plants.splitlines())
                 if re.match(r"^[A-Z][a-z]+ [a-z-]+", l) and not l.endswith("*")}   # "*" = Madeira only
print(f"GIFT: {len(gift):,} species; national invasive list (mainland): {len(invasive_list)} plant names")

In [ ]:
TAXA_FILE = RAW_CS / "inaturalist_taxa_ancestry.csv"
TRACHEOPHYTA, PINOPSIDA = "211194", "136329"   # iNaturalist taxon ids: vascular plants, conifers
if not TAXA_FILE.exists():
    ids, rows = sorted(bg["taxon_id"].dropna().astype(int).unique()), []
    for i in range(0, len(ids), 30):
        r = requests.get("https://api.inaturalist.org/v1/taxa",
                         params=dict(id=",".join(map(str, ids[i:i + 30])), per_page=30), timeout=60)
        r.raise_for_status()
        rows += [dict(taxon_id=t["id"], ancestor_ids=" ".join(map(str, t["ancestor_ids"]))) for t in r.json()["results"]]
        time.sleep(1)
    pd.DataFrame(rows).to_csv(TAXA_FILE, index=False)
ancestors = bg["taxon_id"].map(pd.read_csv(TAXA_FILE).set_index("taxon_id")["ancestor_ids"].str.split())

before = len(bg)
bg = bg[ancestors.apply(lambda a: TRACHEOPHYTA in a)].copy()
print(f"{before:,} → {len(bg):,} (dropped {before - len(bg)} non-vascular: mosses, liverworts, algae)")

# GIFT traits by species (binomial), growth form filled from the genus when missing
words = bg["taxon_name"].str.split()
bg["binomial"], genus = words.str[:2].str.join(" "), words.str[0]
traits = bg[["binomial"]].join(
    gift.drop_duplicates("work_species").set_index("work_species")[list(GIFT_TRAITS.values())], on="binomial")
genus_form = gift.dropna(subset=["growth_form"]).groupby("genus")["growth_form"].agg(lambda s: s.value_counts().index[0])
from_genus = traits["growth_form"].isna() & genus.map(genus_form).notna()
traits["growth_form"] = traits["growth_form"].fillna(genus.map(genus_form))
print(f"growth form from GIFT: species {traits.growth_form.notna().mean() - from_genus.mean():.1%}, "
      f"genus {from_genus.mean():.1%}, none {traits.growth_form.isna().mean():.1%}")

# older names used in the national list
SYNONYMS = {"Hakea decurrens": "Hakea sericea",   # EPPO (2024): the Portuguese "H. sericea" is H. decurrens subsp. physocarpa
            "Paraserianthes lophantha": "Albizzia lophanta", "Albizia julibrissin": "Albizzia julibrissin",
            "Symphyotrichum squamatum": "Aster squamatus", "Erigeron bonariensis": "Conyza bonariensis",
            "Erigeron canadensis": "Conyza canadensis", "Erigeron sumatrensis": "Conyza sumatrensis",
            "Pontederia crassipes": "Eichhornia crassipes", "Cenchrus setaceus": "Pennisetum setaceum",
            "Cenchrus longisetus": "Pennisetum villosum", "Elodea densa": "Egeria densa",
            "Austrocylindropuntia subulata": "Opuntia subulata"}
bg["invasive"] = bg["binomial"].replace(SYNONYMS).isin(invasive_list)

tree, shrub = traits["growth_form"] == "tree", traits["growth_form"] == "shrub"
tall_herb = (traits["max_height"] > 2) & (traits["climber"] != "obligatory")
rules = [("aquatic", traits["aquatic"] == "aquatic"),
         ("eucalyptus", genus == "Eucalyptus"),
         ("pines and other conifers", ancestors.loc[bg.index].apply(lambda a: PINOPSIDA in a) & tree),
         ("oaks", (genus == "Quercus") & tree),
         ("other large invasives", bg["invasive"] & (tree | shrub | tall_herb)),
         ("other trees", tree),
         ("shrubs", shrub)]
bg["group"] = "herbaceous"
for group, rule in reversed(rules):   # reversed, so the first rule that applies wins
    bg.loc[rule, "group"] = group
display(bg["group"].value_counts().to_frame("observations"))
print("large invasives:", bg.loc[bg.group == "other large invasives", "binomial"].value_counts().head(10).to_dict())

**Result.** 323 non-vascular observations dropped (47 353 →
47 030). GIFT gives the growth form of 92.7% of the observations by
species and 5.9% by genus; 1.4% have none (they go to herbaceous).
Two thirds are herbaceous (29 644), then shrubs (10 114) and the large
invasives (3 295: mostly *Cortaderia*, the two *Hakea*, *Arundo*,
*Phytolacca*, *Ailanthus*). The main forest trees are rarely
photographed: 181 eucalyptus and 338 pines and conifers, against 888
oaks. People record flowers and small plants more than the trees that
cover the district.

## 12. Herbaceous plants only where herbs are the top layer; no shrubs under forest

**What:** each observation gets the COS 2023 class it falls in
(`cos2023`, the level-4 code). Herbaceous observations are kept only
in COS classes where herbs are what the pixel shows from above; in the
others they are dropped. Shrubs are dropped under forest (COS 5).
The other groups are kept wherever they are.

| Herbaceous kept (COS 2023) | Herbaceous dropped |
|---|---|
| temporary crops and rice (2.1), crops and pastures mixed with other uses (2.3) | forests (5): the tree canopy covers the herbs |
| pastures (3) | shrubland (6.1.1.1, *matos*): the shrubs cover the herbs |
| agroforestry and silvopastoral land (4, *montado*): sparse trees over a herbaceous layer | buildings, roads, industry, equipment (1, except below): the pixel is a built surface |
| beaches, dunes, rock, sparse vegetation (7) | permanent crops (2.2: vineyards, orchards, olive groves) and greenhouses (2.4): woody crops or plastic |
| wetlands (8) | water (9) |
| green spaces (1.8.1.1), golf courses (1.3.2.3), vacant lots (1.7.1.1) | |

**Why (decided 2026-10-30, to confirm with the supervisor, see
`thesis/notes.md`):** the satellite sees what covers the 10 m pixel.
The FAO Land Cover Classification System decides the dominant life
form by the "uppermost canopy" layer, and treats sparse trees over a
closed herbaceous layer (tree savannah) as herbaceous (Di Gregorio &
Jansen, 2000). A herb photographed under eucalyptus gives a pixel
that, from above, is eucalyptus; labelled "herbaceous", it would teach
the model that forest canopy is herbaceous. Dropping only the herbs
under forest is not enough: herbs on built land would teach the model
that roofs and roads are herbaceous, the same problem the supervisor
pointed out for the expert negatives on buildings.

**Limitations:** COS has a minimum mapping unit of 1 ha, so a small
clearing inside a forest polygon is lost, and a field may hold a few
trees. The list of classes is my reading of the COS class names, not
a published rule. Many pine and eucalyptus stands have an open
canopy, so a pixel inside a COS forest polygon may really be shrubs.

**Shrubs under forest (decided 2026-11-03, see `thesis/notes.md`):**
the same reason as for the herbs: a shrub photographed under pines
gives a pixel that, from above, is pine. This is the starting choice;
the ablation will compare the model with and without this rule.

In [ ]:
COS = ROOT / "data" / "interim" / "spatial" / "COS2023v1-S2.gpkg"
blocks = pd.Series(list(zip(bg.geometry.x // 10_000, bg.geometry.y // 10_000)), index=bg.index)
joined = []
for (bx, by), d in bg.groupby(blocks):  # read COS only around the observations, 10 km at a time
    cos = gpd.read_file(COS, layer="COS2023v1", bbox=(bx * 1e4, by * 1e4, (bx + 1) * 1e4, (by + 1) * 1e4),
                        columns=["COS23_n4_C"])
    joined.append(gpd.sjoin(d, cos, predicate="within", how="left").drop(columns="index_right"))
bg = pd.concat(joined)
bg = bg[~bg.index.duplicated()].rename(columns={"COS23_n4_C": "cos2023"}).sort_index()

def cos_group(code):
    if not isinstance(code, str): return "outside COS"
    if code.startswith("1"): return "artificial"
    if code[0] in "234": return "agriculture and pasture"
    if code == "5.1.1.6": return "eucalyptus forest"
    if code == "5.1.1.7": return "acacia forest"
    if code.startswith("5.1.2"): return "pine forest"
    if code.startswith("5"): return "other forest"
    if code.startswith("6"): return "shrubland"
    return "water, wetlands, bare"
cos_land = bg["cos2023"].map(cos_group)
display(pd.crosstab(bg["group"], cos_land, normalize="index").mul(100).round(1))

# COS 2023 classes where herbs are the top layer (see the table above)
HERB_TOP = ("2.1", "2.3", "3.", "4.", "7.", "8.", "1.8.1.1", "1.3.2.3", "1.7.1.1")
herb_top = bg["cos2023"].fillna("").str.startswith(HERB_TOP)
herb = bg["group"] == "herbaceous"
print("herbaceous by COS level 2:", bg.loc[herb, "cos2023"].fillna("none").str[:3].value_counts().head(8).to_dict())
before = len(bg)
bg = bg[~herb | herb_top]
print(f"{before:,} → {len(bg):,} (dropped {before - len(bg):,} herbaceous where something else covers the pixel)")

# shrubs under forest (COS 5): the tree canopy covers them (same rule as the herbs)
shrub_forest = (bg["group"] == "shrubs") & bg["cos2023"].fillna("").str.startswith("5")
print("shrubs under forest by COS class:", bg.loc[shrub_forest, "cos2023"].map(cos_group).value_counts().to_dict())
before = len(bg)
bg = bg[~shrub_forest]
print(f"{before:,} → {len(bg):,} (dropped {before - len(bg):,} shrubs under forest)")


**Result** (table: share of each group in each COS class, %).
- Herbaceous: 43% are under forest, 12% on buildings and 8% in
  shrubland. 23 677 are dropped and 5 967 stay (47 030 → 23 353).
  Dropping only those under forest would keep 3 523 herbs on
  residential buildings, 2 361 in shrubland and 1 696 in orchards and
  olive groves.
- The tree groups match COS: pines are mostly in pine forest (49%),
  aquatic plants mostly in water or wetlands (44%).
- 23% of the eucalyptus observations fall in COS acacia forest:
  eucalyptus and acacia grow together, so they are the classes most
  likely to be confused.
- Shrubs are mostly under forest (pine 26%, eucalyptus 21%), only 13%
  in shrubland. 6 633 shrubs under forest are dropped (pine 2 604,
  eucalyptus 2 087, other forest 1 623, acacia 319): 23 353 → 16 720.

## 13. One observation per pixel

**What:** where several other plants fall in the same 10 m pixel, keep
one: the tallest group (trees > large invasives > shrubs > herbaceous
> aquatic), then the one closest in date to 2023, then the most recent
(the same date rule as for the presences).

**Why (decided 2026-10-30, to confirm with the supervisor, see
`thesis/notes.md`):** as for the presences, the same pixel is the same
training example (Karasiak et al., 2022). A pixel with a shrub and an
oak would be the same pixel with two different labels; the satellite
sees the tallest plant, the same reason as in step 12. This replaces
the earlier plan of one observation per pixel *and class*, which kept
those conflicting pixels.

In [ ]:
HEIGHT = {"eucalyptus": 0, "pines and other conifers": 0, "oaks": 0, "other trees": 0,
          "other large invasives": 1, "shrubs": 2, "herbaceous": 3, "aquatic": 4}
n_groups = bg.groupby("pixel")["group"].nunique()
print(f"pixels with more than one observation: {bg.pixel.duplicated().sum():,} repeats; "
      f"pixels with more than one group: {(n_groups > 1).sum()}")
before = len(bg)
bg["height"] = bg["group"].map(HEIGHT)
bg["dist_to_ref"] = (bg["year"] - REF_YEAR).abs()
bg = (bg.sort_values(["height", "dist_to_ref", "date"], ascending=[True, True, False])
        .drop_duplicates("pixel").drop(columns=["height", "dist_to_ref"]).sort_index())
print(f"{before:,} → {len(bg):,} (one per pixel)")
print(f"other plants per A. dealbata presence: {len(bg) / len(acacia):.1f}; Observer A: {(bg.observer == 'Observer A').mean():.1%}")
bg["group"].value_counts().to_frame("observations")

**Result.** 16 720 → 13 554. 646 pixels held more than one group.
Final classes: herbaceous 4 229, large invasives 2 958, shrubs 2 790,
other trees 1 663, oaks 802, aquatic 624, pines 319, eucalyptus 169.
4.5 other plants per *A. dealbata* presence; Observer A made 29.7% of
them. Eucalyptus and pines are small classes; whether to merge them
is decided when training (class weighting is still open).

## 14. Burned-area flag

Same rule and code as for *Acacia* (`src/burned_areas.py`): `burned`
is `True` if the observation lies in an area that burned between the
observation and 2023. Nothing is removed.

In [ ]:
fires = load_burned_areas(ROOT / "data" / "external" / "auxiliary" / "icnf_burned_areas")
fires = fires[fires.intersects(district)]
bg["burned"], bg["burned_same_year"] = burned_between(bg, fires, REF_YEAR)
print(f"burned between observation and {REF_YEAR}: {bg.burned.sum():,} ({bg.burned.mean():.1%}); "
      f"of which same year without a fire date: {bg.burned_same_year.sum()}")

**Result.** 125 observations (0.9%) are flagged, more than for
*A. dealbata* (0.2%): other plants include older records, and the
2017 fires burned after many of them. No case is ambiguous.

## 15. Save

In [ ]:
OUT = INTERIM_CS / "inat_other_plants_clean.csv"
keep = ["id", "observed_on", "year", "taxon_id", "taxon_name", "taxon_rank", "latitude", "longitude",
        "positional_accuracy", "quality_grade", "accuracy_missing", "observer", "pixel", "group", "cos2023", "burned", "burned_same_year"]
bg[keep].to_csv(OUT, index=False)
print(OUT.relative_to(ROOT), f"{len(bg):,} observations")

## Summary

**Download.** 120 479 plant observations that are not *Acacia*,
1976–2026; 9 903 with obscured coordinates.

**Part 1 — exploration.** 110 576 dated, non-obscured observations,
32 per *A. dealbata* presence (research grade). They cover the places
where *A. dealbata* was seen (96.6% of its 1 km cells) and much more
(only 16.3% of their cells have *A. dealbata*). Observer A made 33% of
them.

**Part 2 — cleaning.** Obscured coordinates dropped; main observer
anonymised; only research grade kept, the same rule as for the
presences (110 576 → 60 630); positional accuracy above 30 m dropped,
missing values kept and marked, also the same rule (60 630 → 48 125);
other plants in the 443 pixels that also hold an *A. dealbata*
presence dropped (48 125 → 47 353); species grouped into 8 classes from GIFT growth forms, the national
invasive list and the iNaturalist taxonomy, non-vascular plants
dropped (→ 47 030); herbaceous plants kept only in COS classes where
herbs are the top layer (→ 23 353); shrubs under forest dropped
(→ 16 720); one observation per pixel, the tallest group first
(→ 13 554, 4.5 per presence, Observer A 29.7%); `burned` flag added
(125, 0.9%; nothing removed). Result: 13 554
observations in `data/interim/citizen_science/inat_other_plants_clean.csv`,
with `group` and `cos2023`. Steps 11–13 are proposals to confirm with
the supervisor. Future work: sensitivity check with and without `casual` and
`needs_id`, on both sides.